In [1]:
import os
import io
import json
import yaml
import docling
import ollama
import base64
from pathlib import Path
import logging

In [2]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker


In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0

In [4]:
converter= DocumentConverter(

    format_options={
    InputFormat.PDF: PdfFormatOption(
        pipeline_options=pipeline_options
    )
    }
)

In [5]:
DATA_FOLDER = "data"  # Update this path to your PDF folder
existing_ids = []  # Add logic to track already indexed documents if needed

for file in os.listdir(DATA_FOLDER):
    if not file.endswith(".pdf"):
        continue
    if any(file in x for x in existing_ids):
        print(f"Skipping {file} (already indexed)")
        continue

    print(f"\nIndexing: {file}")

    try:
        result = converter.convert(os.path.join(DATA_FOLDER, file))
        doc = result.document
      
        #Export to word
        display_markdown_output_path = Path("document_output.md")
        with display_markdown_output_path.open("w", encoding="utf-8") as f:
            f.write(doc.export_to_markdown())
        print(f"Saved Markdown: {display_markdown_output_path}")
        # Save images  
        images_folder = Path("extracted_images")
        images_folder.mkdir(exist_ok=True)

        for i, picture in enumerate(doc.pictures):
            try:
                pil_img = picture.image.pil_image
                if pil_img is None:
                    continue
                page = picture.prov[0].page_no if picture.prov else "unknown"
                img_path = images_folder / f"{file.replace('.pdf', '')}_page{page}_img{i}.png"
                pil_img.save(img_path)
                print(f"  Saved image {i} from page {page}")
            except Exception as e:
                print(f"  Image {i} failed: {e}")
            
    except Exception as e:
        print(f"Error processing {file}: {e}")


Indexing: APDFOFTHINGS.pdf


[INFO] 2026-06-02 10:19:03,684 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-02 10:19:03,745 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\manya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-02 10:19:03,747 [RapidOCR] main.py:57: Using C:\Users\manya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-02 10:19:04,005 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-02 10:19:04,059 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\manya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-02 10:19:04,060 [RapidOCR] main.py:57: Using C:\Users\manya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-02 10:19:04,167 [RapidOCR] base

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

RapidOCR returned empty result!
[WARNING] 2026-06-02 10:19:11,146 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!


Saved Markdown: document_output.md
  Saved image 0 from page 2
  Saved image 1 from page 2
  Saved image 2 from page 3
  Saved image 3 from page 3
  Saved image 4 from page 3
  Saved image 5 from page 3
  Saved image 6 from page 3


Langchain chunker

In [6]:
!pip install langchain-text-splitters

from langchain_text_splitters import MarkdownTextSplitter

C:\Users\manya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [7]:


splitter = MarkdownTextSplitter(chunk_size=1000, chunk_overlap=100)
with open("document_output.md", "r", encoding="utf-8") as f:
    md_text = f.read()

chunks = splitter.split_text(md_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 9


In [8]:
print(f"\n{'='*60}")
print(f"Total Chunks: {len(chunks)}")
print(f"{'='*60}")
for i, chunk in enumerate(chunks):
    preview = chunk[:100].replace('\n', ' ')[:80] + "..."
    print(f"Chunk {i:2d} | {len(chunk):4d} chars | {preview}")


Total Chunks: 9
Chunk  0 |   54 chars | ## ABOUT THE WORLD  ## 1. Historical Overview of Syria...
Chunk  1 |  995 chars | Syria is one of the oldest continuously inhabited regions in the world, with evi...
Chunk  2 |  997 chars | philosophy, and early Christianity. Syria's location at the crossroads of Europe...
Chunk  3 |  848 chars | collapse of the Ottoman Empire after World War I, Syria became a French mandate ...
Chunk  4 |  713 chars | <!-- image -->  <!-- image -->  <!-- image -->  ## Tokenization  The input text ...
Chunk  5 |  917 chars | | Plant Type        | Characteristics                                      | Exa...
Chunk  6 |  889 chars | ## 3. Hierarchical Discussion on China  ## 3.1 Ancient China  Ancient China deve...
Chunk  7 |  861 chars | ## 3.2.1 Trade and Innovation  China played a central role in Eurasian trade net...
Chunk  8 |  497 chars | ## 3.3.2 Global Influence  China today plays a major role in international trade...


VECTORDB

In [14]:
import chromadb

# Connect to ChromaDB (chunks already created in previous cell)
client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_or_create_collection(name="documents")

# Add chunks to collection
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"source": "APDFOFTHINGS.pdf", "chunk_id": i} for i in range(len(chunks))]
)

# Verify
print(f"✓ New documents indexed in ChromaDB: {collection.count()} total chunks")

✓ New documents indexed in ChromaDB: 9 total chunks


In [10]:
import chromadb

# Connect to ChromaDB
client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_or_create_collection(name="documents")

# DELETE all existing chunks
existing_ids = collection.get()["ids"]
if existing_ids:
    collection.delete(ids=existing_ids)
    print(f"✓ Deleted {len(existing_ids)} existing chunks")
else:
    print("✓ No existing chunks to delete")

# Show collection is now empty
print(f"  Collection now contains: {collection.count()} documents")

✓ Deleted 52 existing chunks
  Collection now contains: 0 documents


In [17]:
import chromadb
import os
from functools import lru_cache
import json
import time

# Setup (run once)
client = chromadb.PersistentClient(path="./chromadb")
collection = client.get_or_create_collection(name="documents")

# Initialize LLM - using Google Gemini API
if 'llm' in globals():
    del llm

llm = None
print("Initializing LLM...")

# Use Google Gemini API
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    api_key = os.getenv("GOOGLE_API_KEY", "")
    if api_key:
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            google_api_key=api_key,
            max_retries=3,
            request_timeout=30
        )
        print("✓ Using Google Gemini 2.0 Flash (Limited free tier ~60 requests/day)")
    else:
        print("❌ GOOGLE_API_KEY environment variable not set")
        print("   Set it with: $env:GOOGLE_API_KEY='your-api-key'")
except Exception as e:
    print(f"❌ Failed to initialize Gemini: {e}")

# Load or create response cache (to avoid re-querying)
CACHE_FILE = "query_cache.json"
try:
    with open(CACHE_FILE, "r") as f:
        query_cache = json.load(f)
except:
    query_cache = {}

# Track last API call time for rate limiting
last_api_call = {"time": 0}

def query_rag(query, use_cache=True, n_results=2, chunk_chars=400):
    """Query RAG with optimizations to reduce API usage
    
    Args:
        query: Question to ask
        use_cache: Check cache first (saves API calls)
        n_results: Number of chunks to retrieve (lower = faster)
        chunk_chars: Max characters to send (400-500 chars)
    """
    
    # CHECK CACHE FIRST - No API call needed!
    if use_cache and query in query_cache:
        print("[CACHED]", end=" ")
        return query_cache[query]
    
    if llm is None:
        return "Error: No LLM available. Set GOOGLE_API_KEY environment variable"
    
    # RATE LIMITING: Wait to avoid 429 errors
    time_since_last_call = time.time() - last_api_call["time"]
    min_delay = 4  # 4 second gap = ~15 RPM (safe for free tier)
    if time_since_last_call < min_delay:
        wait_time = min_delay - time_since_last_call
        print(f"[RATE LIMIT] Waiting {wait_time:.1f}s...", end=" ")
        time.sleep(wait_time)
    
    # Retrieve context chunks
    result = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    retrieved_docs = result["documents"][0]
    retrieved_metas = result["metadatas"][0]

    # OPTIMIZATION: Truncate chunks to reduce processing
    context = "\n\n---\n\n".join(
        f"Source: {meta.get('source', 'unknown')}\n{doc[:chunk_chars]}"
        for doc, meta in zip(retrieved_docs, retrieved_metas)
    )

    # OPTIMIZATION: Shorter prompt
    prompt = f"""Answer based on this context only.
Query: {query}
Context: {context}
Answer:"""

    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, 'content') else str(response)
    
    # Record API call time
    last_api_call["time"] = time.time()
    
    # Save to cache
    query_cache[query] = answer
    with open(CACHE_FILE, "w") as f:
        json.dump(query_cache, f, indent=2)
    
    return answer


# Example query
test_query = "What is the main topic of the PDF?"
answer = query_rag(test_query)
print(f"Query: {test_query}")
print(f"Answer: {answer}")
print("-" * 50)

Initializing LLM...
✓ Using Google Gemini 2.0 Flash (Limited free tier ~60 requests/day)
[CACHED] Query: What is the main topic of the PDF?
Answer: The main topic of this PDF is about Syria's historical overview and global influences. The PDF also briefly mentions China's participation in international organizations and initiatives. The information presented is divided into sections with mixed formatting structures such as numbered headingings and nested paragraphs.
--------------------------------------------------


In [18]:
# Quick query template - modify and rerun this cell
my_query = "tell me ancient china"
result = query_rag(my_query, use_cache=True, n_results=2, chunk_chars=500)
print(f"\n{'='*60}")
print(f"Q: {my_query}")
print(f"{'='*60}")
print(f"A: {result}")
print(f"{'='*60}")


Q: tell me ancient china
A: Ancient China developed along the Yellow River basin and gave rise to some of the earliest dynasties in world history, including the Xia, Shang, and Zhou. These dynasties contributed to the formation of early Chinese political systems, bronze technology, and philosophical traditions.

During the Zhou period, classical Chinese philosophy flourished, with thinkers such as Confucius and Laozi establishing influential schools of thought.

China also played a central role in Eurasian trade networks through the Silk Road. Innovations like papermaking, gunpowder, the magnetic compass, and movable-type printing, which had lasting global impacts, originated there.


In [20]:
my_query = "details on plant types"
result = query_rag(my_query, use_cache=True, n_results=2, chunk_chars=500)
print(f"\n{'='*60}")
print(f"Q: {my_query}")
print(f"{'='*60}")
print(f"A: {result}")
print(f"{'='*60}")


Q: details on plant types
A: Based on the context provided, here are the details on plant types:

*   **Emergent Trees:** These are very tall trees that rise above the canopy layer. Examples include the Kapok Tree and Brazil Nut Tree.
*   **Canopy Plants:** This type consists of dense upper-layer vegetation that receives most of the sunlight. Examples are the Rubber Tree and Mahogany.
*   **Understory Plants:** These are shade-tolerant plants that grow beneath the canopy.
